## Power Plant Energy Output Prediction Using Regression
The goal of this project is to predict the electrical power output (PE)
of a combined cycle power plant using environmental variables:

- Ambient Temperature (AT)
- Exhaust Vacuum (V)
- Ambient Pressure (AP)
- Relative Humidity (RH)

Because PE is a continuous target variable and labeled training data are available, the task is a supervised learning regression problem.

## Import Libraries & Packages 

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

## Exploratory Data Analysis (EDA)

In [4]:
df = pd.read_csv("data/CCPP_data.csv")

print(df.head())
print(df.info())
print(df.describe())
print(df.isnull().sum())

      AT      V       AP     RH      PE
0  14.96  41.76  1024.07  73.17  463.26
1  25.18  62.96  1020.04  59.08  444.37
2   5.11  39.40  1012.16  92.14  488.56
3  20.86  57.32  1010.24  76.64  446.48
4  10.82  37.50  1009.23  96.62  473.90
<class 'pandas.DataFrame'>
RangeIndex: 9568 entries, 0 to 9567
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   AT      9568 non-null   float64
 1   V       9568 non-null   float64
 2   AP      9568 non-null   float64
 3   RH      9568 non-null   float64
 4   PE      9568 non-null   float64
dtypes: float64(5)
memory usage: 373.9 KB
None
                AT            V           AP           RH           PE
count  9568.000000  9568.000000  9568.000000  9568.000000  9568.000000
mean     19.651231    54.305804  1013.259078    73.308978   454.365009
std       7.452473    12.707893     5.938784    14.600269    17.066995
min       1.810000    25.360000   992.890000    25.560000   420.260000
25% 

## Step 1: Define Features, Target Variable, and Evaluation Metrics

The dataset is used to predict electrical power output (PE) using environmental measurements.

Input features (X): environmental variables
Target variable (y): electrical power output (PE)

In [5]:
X = df.drop(columns=["PE"])
y = df["PE"]

## Machine Learning Approach

Since PE is a continuous numerical target variable and labeled training data are available, this task is a supervised machine learning regression problem.

### Evaluation Metrics

To assess model performance, the following regression evaluation metrics are used:

- Mean Absolute Error (MAE): measures the average absolute difference between actual and predicted values.
- Root Mean Squared Error (RMSE): measures prediction error while penalizing larger errors more heavily.
- R² Score (Coefficient of Determination): indicates how well the model explains the variance in the target variable.

These metrics provide complementary insights into prediction accuracy and overall model performance.

## Step 2: Identify Features and Possible Algorithms

### Feature Variables

The input features used to predict electrical power output (`PE`) are:

- Ambient Temperature (`AT`)
- Exhaust Vacuum (`V`)
- Ambient Pressure (`AP`)
- Relative Humidity (`RH`)

In [7]:
print(X.columns)

Index(['AT', 'V', 'AP', 'RH'], dtype='str')


## Candidate Regression Algorithms

### Possible machine learning algorithms for this regression task include:

1. Linear Regression
2. Ridge Regression
3. Random Forest Regression

### Algorithm Selection Rationale

- Linear-based models are simple and interpretable.
- Ridge Regression helps reduce overfitting by adding regularization.
- Random Forest Regression can model nonlinear relationships and interactions between variables.

## Step 3: Split the Data and Define Validation Strategy
### Train-Test Split

The dataset is divided into:

- 80% training data
- 20% testing data

The training set is used to train the models, while the testing set is reserved for final evaluation.

## Step 4: Create Train/Test Split

In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)

Training set: (7654, 4)
Test set: (1914, 4)


## Step 5:  Compare Models Using Cross-Validation 

In [ ]:
models = {
    "Linear Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LinearRegression())
    ]),
    
    "Ridge Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", Ridge())
    ]),
    
    "Random Forest": RandomForestRegressor(random_state=42)
}

for name, model in models.items():
    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=5,
        scoring="r2"
    )
    print(name)
    print("CV R2 scores:", scores)
    print("Mean CV R2:", scores.mean())
    print()

Linear Regression
CV R2 scores: [0.92603255 0.92407087 0.93306007 0.92934518 0.92862547]
Mean CV R2: 0.9282268261521758

Ridge Regression
CV R2 scores: [0.92604206 0.92406374 0.93305546 0.92934799 0.92862513]
Mean CV R2: 0.92822687547353

Random Forest
CV R2 scores: [0.95822288 0.95405781 0.96465324 0.95835531 0.95876102]
Mean CV R2: 0.9588100513200777



## Model Comparison Summary

Three regression algorithms were compared using 5-fold cross-validation.

- Linear Regression and Ridge Regression produced similar performance with mean R² scores around 0.928.
- Random Forest Regression achieved the highest validation performance with a mean R² score of approximately 0.959.

Because Random Forest Regression produced the best validation results, it was selected for hyperparameter tuning and final evaluation.

In [17]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestRegressor

## Step 6: Tune Random Forest Hyperparameters

In [19]:
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5]
}

grid_search = GridSearchCV(
    RandomForestRegressor(random_state=42),
    param_grid,
    cv=5,
    scoring="r2",
    n_jobs=1
)

grid_search.fit(X_train, y_train)

print("Best parameters:", grid_search.best_params_)
print("Best validation R2:", grid_search.best_score_)

Best parameters: {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 200}
Best validation R2: 0.9591381993342163


## Step 7: Select Final Model 

In [20]:
final_model = grid_search.best_estimator_

## Step 8: Evaluate on Test Set

In [21]:
y_pred = final_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("Final Test Results")
print("MAE:", mae)
print("RMSE:", rmse)
print("R2 Score:", r2)

Final Test Results
MAE: 2.3146691222570515
RMSE: 3.230977449275154
R2 Score: 0.9640099014655094


## Final Model Performance Interpretation

The tuned Random Forest Regression model achieved strong predictive performance on the test set.

- The R² score of 0.964 indicates that approximately 96.4% of the variance in electrical power output is explained by the model.
- The RMSE value of 3.23 suggests that prediction errors are relatively small.
- The MAE of 2.31 indicates that the average prediction error is about 2.31 units of power output.

Overall, Random Forest Regression outperformed the linear models and was selected as the final model.

## Step 9: Compare Actual vs Predicted 

In [22]:
results = pd.DataFrame({
    "Actual PE": y_test.values,
    "Predicted PE": y_pred
})

print(results.head(10))

   Actual PE  Predicted PE
0     455.27     455.55325
1     436.31     435.91535
2     440.68     434.80485
3     434.40     435.18065
4     482.06     478.73555
5     436.07     435.71670
6     452.48     451.37010
7     435.22     434.87035
8     432.93     435.04810
9     466.46     468.02005
